In [43]:
import csv
import json
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch

In [44]:
from flowrec.data.yambda import YambdaDataConfig, YambdaDataModule
from flowrec.models.popularity import PopularityConfig, PopularityModel
from flowrec.models.svd import SVDConfig, SVDModel
SEEDS = (42, 43, 44)

In [45]:
YAMBDA_DATASETS = {
    "yambda_50m_listens": {
        "use_hf": True,
        "dataset_type": "sequential",
        "dataset_size": "50m",
        "event_file": "listens",
    },
}

In [46]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


In [47]:
def make_yambda_config(dataset_cfg: dict[str, Any], seed: int) -> YambdaDataConfig:
    kwargs = dict(dataset_cfg)

    return YambdaDataConfig(
        **kwargs,

        # Common data config.
        split_strategy="global_time",
        global_val_ratio=0.1,
        global_test_ratio=0.1,
        global_eval_target_mode="last_per_user",
        global_history_policy="rolling",
        min_played_ratio_pct=50,

        # Requested setup.
        train_all_prefixes=False,

        # Yambda filtering.
        min_user_interactions=10,
        min_item_interactions=10,
        max_history_len=128,

        # Batching.
        batch_size=256,
        eval_batch_size=256,
        num_workers=0,
        seed=seed,
    )


In [48]:
def fit_popularity_model(dm) -> PopularityModel:
    num_items = dm.item_vocab_size

    model = PopularityModel(
        PopularityConfig(
            num_items=num_items,
            pad_item_id=0,
            score_transform="log1p",
            exclude_history_in_generate=True,
        )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None


    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        try:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets_only=False,
            )
        except TypeError:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets=True,
                count_history=True,
            )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_targets=True,
            count_history=True,
        )

    return model


def fit_svd_model(dm, seed: int) -> SVDModel:
    num_users = dm.num_users
    num_items = dm.item_vocab_size

    model = SVDModel(
        SVDConfig(
            num_users=num_users,
            num_items=num_items,
            n_components=256,
            pad_item_id=0,
            input_transform="binary",

            # Recommended for next-item prediction:
            user_representation="history_foldin",
            foldin_reg=1e-2,

            exclude_history_in_generate=True,
            score_chunk_size=50_000,
            random_state=seed,
        )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None

    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        model.fit_from_store(
            store=store,
            target_index=train_index,
            fit_source="observed_prefix",
            use_ratings=False,
        )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_history=True,
            count_targets=True,
            use_ratings=False,
        )

    return model

In [49]:
@torch.no_grad()
def evaluate_model(
    model,
    test_loader,
    *,
    num_items: int,
    ks: tuple[int, ...] = (5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    model.eval()

    max_k = max(ks)

    total = 0
    hit_sum = {k: 0.0 for k in ks}
    ndcg_sum = {k: 0.0 for k in ks}
    coverage_mask = {k: torch.zeros(num_items, dtype=torch.bool) for k in ks}

    for batch in test_loader:
        target = batch.get("target_item_ids", batch.get("target_item_id"))

        if target is None:
            raise KeyError("Batch must contain target_item_ids or target_item_id.")

        target = target.long()

        recs = model.generate(
            batch,
            num_return_sequences=max_k,
        ).cpu()

        valid = target != pad_item_id
        valid &= target >= 0
        valid &= target < num_items

        if valid.sum() == 0:
            continue

        target = target[valid].cpu()
        recs = recs[valid]

        batch_size = target.size(0)
        total += batch_size

        for k in ks:
            topk = recs[:, :k]

            matches = topk.eq(target.unsqueeze(1))
            hit = matches.any(dim=1)

            hit_sum[k] += float(hit.float().sum().item())

            rank = matches.float().argmax(dim=1)
            ndcg = torch.zeros(batch_size, dtype=torch.float32)

            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank[hit].float() + 2.0)

            ndcg_sum[k] += float(ndcg.sum().item())

            valid_recs = topk[
                (topk != pad_item_id)
                & (topk >= 0)
                & (topk < num_items)
            ]

            if valid_recs.numel() > 0:
                coverage_mask[k][valid_recs.unique()] = True

    if total == 0:
        raise RuntimeError("No valid test examples found.")

    catalog_size = max(num_items - 1, 1)

    metrics: dict[str, float] = {}

    for k in ks:
        metrics[f"hit_rate@{k}"] = hit_sum[k] / total
        metrics[f"ndcg@{k}"] = ndcg_sum[k] / total
        metrics[f"coverage@{k}"] = float(coverage_mask[k][1:].float().mean().item())

    metrics["num_test_examples"] = float(total)
    metrics["num_items"] = float(catalog_size)

    return metrics


In [50]:
def run_one_model(
    *,
    model_name: str,
    dm,
    seed: int,
) -> dict[str, float]:
    if model_name == "most_popular":
        model = fit_popularity_model(dm)
    elif model_name == "svd":
        model = fit_svd_model(dm, seed=seed)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    metrics = evaluate_model(
        model,
        dm.test_dataloader(),
        num_items=dm.item_vocab_size,
        ks=(5, 10, 20),
        pad_item_id=0,
    )

    return metrics


In [51]:
def run_one_seed(
    dataset_name: str,
    dataset_cfg: dict[str, Any],
    seed: int,
) -> list[dict[str, Any]]:
    print(f"\n=== Dataset={dataset_name} | seed={seed} ===")

    set_seed(seed)

    data_cfg = make_yambda_config(dataset_cfg, seed)

    dm = YambdaDataModule(data_cfg)
    dm.setup()

    print(json.dumps(dm.summary(), indent=2))

    rows: list[dict[str, Any]] = []

    for model_name in ("most_popular", "svd"):
        print(f"Running model: {model_name}")

        try:
            metrics = run_one_model(
                model_name=model_name,
                dm=dm,
                seed=seed,
            )
            print(metrics)

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                **metrics,
            }

        except MemoryError as exc:
            print(f"Skipping {model_name} due to MemoryError: {exc}")
            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                "error": f"MemoryError: {exc}",
            }

        rows.append(row)

    return rows


def aggregate_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    metric_keys = [
        "hit_rate@5",
        "hit_rate@10",
        "hit_rate@20"
        "ndcg@5",
        "ndcg@10",
        "ndcg@20",
        "coverage@5",
        "coverage@10",
        "coverage@20",
        "num_test_examples",
        "num_items",
    ]

    groups: dict[tuple[str, str], list[dict[str, Any]]] = {}

    for row in rows:
        if "error" in row:
            continue
        key = (row["dataset"], row["model"])
        groups.setdefault(key, []).append(row)

    summary_rows: list[dict[str, Any]] = []

    for (dataset, model), group_rows in groups.items():
        summary: dict[str, Any] = {
            "dataset": dataset,
            "model": model,
            "seed": "mean",
        }

        for metric in metric_keys:
            values = [
                float(row[metric])
                for row in group_rows
                if metric in row
            ]
            if not values:
                continue

            values_np = np.asarray(values, dtype=np.float64)
            summary[metric] = float(values_np.mean())
            summary[f"{metric}_std"] = float(values_np.std(ddof=0))

        summary_rows.append(summary)

    return summary_rows


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: list[str] = []

    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [52]:
output_dir = Path("../artifacts/metrics/yambda_baselines")
output_dir.mkdir(parents=True, exist_ok=True)

per_seed_rows: list[dict[str, Any]] = []

for dataset_name, dataset_cfg in YAMBDA_DATASETS.items():
    for seed in SEEDS:
        per_seed_rows.extend(
            run_one_seed(
                dataset_name=dataset_name,
                dataset_cfg=dataset_cfg,
                seed=seed,
            )
        )

summary_rows = aggregate_rows(per_seed_rows)

write_csv(output_dir / "per_seed_metrics.csv", per_seed_rows)
write_csv(output_dir / "summary_metrics.csv", summary_rows)

print(f"\nSaved per-seed metrics to: {output_dir / 'per_seed_metrics.csv'}")
print(f"Saved summary metrics to: {output_dir / 'summary_metrics.csv'}")


=== Dataset=yambda_50m_listens | seed=42 ===
{
  "num_users": 9033,
  "num_items": 194307,
  "num_train_examples": 8417,
  "num_val_examples": 8079,
  "num_test_examples": 8814,
  "num_interactions": 28276618,
  "dataset": "yambda",
  "dataset_type": "sequential",
  "dataset_size": "50m",
  "event_file": "listens",
  "split_strategy": "global_time",
  "global_val_cutoff": 22139365,
  "global_test_cutoff": 24141315,
  "sid_len": null,
  "item_vocab_size": 194308
}
Running model: most_popular
{'hit_rate@5': 0.0038574994327206717, 'ndcg@5': 0.0023333062883783475, 'coverage@5': 0.00015954134869389236, 'hit_rate@10': 0.006466984343090538, 'ndcg@10': 0.0031475782164484298, 'coverage@10': 0.00022644577256869525, 'hit_rate@20': 0.009643748581801679, 'ndcg@20': 0.003940359194153746, 'coverage@20': 0.000349961657775566, 'num_test_examples': 8814.0, 'num_items': 194307.0}
Running model: svd
{'hit_rate@5': 0.009303380984796914, 'ndcg@5': 0.005482442820724079, 'coverage@5': 0.010143741965293884, '

| Dataset            | Model        | Hit Rate@5 | Hit Rate@10 | Hit Rate@20 | NDCG@5   | NDCG@10  | NDCG@20  | Coverage@5 | Coverage@10 | Coverage@20 | Num Test Examples | Num Items |
| ------------------ | ------------ | ---------- | ----------- | ----------- | -------- | -------- | -------- | ---------- | ----------- | ----------- | ----------------- | --------- |
| yambda_50m_listens | most_popular | 0.003857   | 0.006467    | 0.009644    | 0.002333 | 0.003148 | 0.003940 | 0.000160   | 0.000226    | 0.000350    | 8,814             | 194,307   |
| yambda_50m_listens | svd          | 0.009303   | 0.015543    | 0.023939    | 0.005482 | 0.007501 | 0.009591 | 0.010144   | 0.013772    | 0.018383    | 8,814             | 194,307   |
